# 03 Newton 法、阻尼 Newton 与重根修正

前两节分别强调了可靠的括区间思想和不动点迭代的局部收敛条件。Newton 法把这两条线索连接起来：它仍可看成不动点迭代

$$
x_{k+1}=g(x_k)=x_k-\frac{f(x_k)}{f'(x_k)},
$$

但这个 $g$ 来自 $f$ 在当前点的一阶 Taylor 线性化。若 $\alpha$ 是简单根，且 $f'(\alpha)\ne 0$，Newton 法在根附近通常具有二次收敛。远离根时，它可能因为导数很小、步长过大或函数非凸而失败，因此实际算法常加入阻尼和回溯保护。


In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
while ROOT.name != "py-sc" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import damped_newton_method, modified_newton_method, newton_method


## Newton 迭代和局部二次收敛

在 $x_k$ 处作一阶 Taylor 展开：

$$
f(x)\approx f(x_k)+f'(x_k)(x-x_k).
$$

令右端为零，就得到 Newton 更新

$$
x_{k+1}=x_k-\frac{f(x_k)}{f'(x_k)}.
$$

对简单根 $\alpha$，误差满足近似关系

$$
e_{k+1}\approx \frac{f''(\alpha)}{2f'(\alpha)}e_k^2,
$$

这解释了为什么足够接近根时有效数字会快速翻倍。


In [ ]:
def teaching_newton(f, df, x0, tolerance=1e-12, max_iterations=20):
    history = [float(x0)]
    x_old = float(x0)
    for k in range(1, max_iterations + 1):
        fx = float(f(x_old))
        dfx = float(df(x_old))
        if abs(fx) <= tolerance:
            return x_old, k - 1, True, np.array(history)
        if abs(dfx) <= 1e-14:
            raise ValueError("derivative is too small")
        x_new = x_old - fx / dfx
        history.append(x_new)
        if abs(f(x_new)) <= tolerance:
            return x_new, k, True, np.array(history)
        x_old = x_new
    return x_old, max_iterations, False, np.array(history)

f = lambda x: x**2 - 2.0
df = lambda x: 2.0 * x
exact = math.sqrt(2.0)
root, iterations, converged, history = teaching_newton(f, df, 1.5)
official = newton_method(f, df, 1.5, tolerance=1e-12)
errors = np.abs(history - exact)
ratios = errors[2:] / errors[1:-1] ** 2

print(f"teaching root = {root:.15f}, iterations = {iterations}, converged = {converged}")
print(f"official root = {official.root:.15f}, residual = {official.residual:.3e}")
print("errors:", np.array2string(errors, precision=3))
print("quadratic ratios:", np.array2string(ratios, precision=3))


## 阻尼 Newton：把过大步长拉回来

Newton 步长来自局部线性模型。若初值远离根，线性模型可能给出过大的步长。阻尼 Newton 使用

$$
x_{k+1}=x_k+\lambda_k p_k,\qquad p_k=-\frac{f(x_k)}{f'(x_k)},\quad 0<\lambda_k\le 1,
$$

并通过回溯寻找能降低残差的 $\lambda_k$。这样会牺牲局部速度，但能减少远离根时的发散风险。


In [ ]:
def first_newton_trial(f, df, x0):
    step = -f(x0) / df(x0)
    return x0 + step, step

cubic = lambda x: x**3 - 1.0
cubic_derivative = lambda x: 3.0 * x**2
trial, step = first_newton_trial(cubic, cubic_derivative, 0.1)
damped = damped_newton_method(cubic, cubic_derivative, 0.1, tolerance=1e-12)

print(f"undamped first trial = {trial:.6f}, first step = {step:.6f}")
print(f"damped root = {damped.root:.15f}, iterations = {damped.iterations}, residual = {damped.residual:.3e}")
print("damped history:", np.array2string(damped.history[:6], precision=8))


## 重根会削弱 Newton 收敛

若 $\alpha$ 是 $m$ 重根，普通 Newton 法的局部收敛会从二次退化为线性。若已知重数 $m$，可以改用

$$
x_{k+1}=x_k-m\frac{f(x_k)}{f'(x_k)}.
$$

这称为重根修正 Newton 法。它的限制也很明显：实际问题中重数常常未知，且重根附近 $f'(x)$ 很小，舍入误差更容易被放大。


In [ ]:
multiple_f = lambda x: (x - 2.0) ** 3
multiple_df = lambda x: 3.0 * (x - 2.0) ** 2
plain_multiple = newton_method(multiple_f, multiple_df, 3.5, tolerance=1e-8, max_iterations=100)
modified_multiple = modified_newton_method(multiple_f, multiple_df, 3.5, multiplicity=3, tolerance=1e-12)

print(f"plain Newton root = {plain_multiple.root:.12f}, iterations = {plain_multiple.iterations}")
print(f"modified Newton root = {modified_multiple.root:.12f}, iterations = {modified_multiple.iterations}")
print("plain residual:", f"{plain_multiple.residual:.3e}", "modified residual:", f"{modified_multiple.residual:.3e}")


## 失败情形

Newton 类方法的常见失败原因包括：

* 当前点导数为零或接近零；
* 初值远离目标根，切线交点落到不合适区域；
* 函数在迭代路径上不可导或数值不稳定；
* 重根附近导数变小，普通 Newton 法明显变慢；
* 阻尼回溯无法找到降低残差的步长。

这些失败并不意味着 Newton 法不好，而是说明它需要与括区间、初值扫描、阻尼策略或重根信息配合使用。


In [ ]:
try:
    newton_method(lambda x: x**3 + 1.0, lambda x: 3.0 * x**2, 0.0)
except ValueError as exc:
    print("zero derivative failure:", exc)


## 本节小结

Newton 法通过局部线性化构造迭代，简单根附近通常二次收敛，因此比二分法和普通不动点迭代快得多。它的代价是需要导数，并且只具有局部可靠性。阻尼 Newton 用回溯降低残差，增强远离根时的稳健性；重根修正 Newton 在已知重数时能恢复多重根附近的收敛速度。实际使用时，常先用扫描或括区间方法找到合理初值，再用 Newton 类方法加速收敛。
